In [ ]:
from pathlib import Path
import shutil

import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"
REY_DIR = DATA_DIR / "REY_DATASET"
REY_SCAN_DIR = REY_DIR / "REY_scan_anonim"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"

YOLO_DIR = BASE_DIR / "yolo"
YOLO_REY_DIR = YOLO_DIR / "rey_bbox"

CSV_LIMPIO = TABLES_DIR / "rey_traza_limpia.csv"

print("CSV limpio existe:", CSV_LIMPIO.exists())
print("REY_SCAN_DIR existe:", REY_SCAN_DIR.exists())
print("YOLO_REY_DIR:", YOLO_REY_DIR)

In [ ]:
df = pd.read_csv(CSV_LIMPIO)

print("Número de muestras:", len(df))
display(df.head())

In [ ]:
if YOLO_REY_DIR.exists():
    shutil.rmtree(YOLO_REY_DIR)

carpetas = [
    YOLO_REY_DIR / "images" / "train",
    YOLO_REY_DIR / "images" / "val",
    YOLO_REY_DIR / "images" / "test",
    YOLO_REY_DIR / "labels" / "train",
    YOLO_REY_DIR / "labels" / "val",
    YOLO_REY_DIR / "labels" / "test",
]

for carpeta in carpetas:
    carpeta.mkdir(parents=True, exist_ok=True)

print("Carpetas YOLO creadas.")

In [ ]:
RANDOM_STATE = 7

df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=RANDOM_STATE
)

print("Train:", len(df_train))
print("Val:", len(df_val))
print("Test:", len(df_test))

In [ ]:
def bbox_a_yolo(fila):
    """
    Convierte la BB del CSV [x, y, h, w]
    a formato YOLO:
    clase x_center y_center width height
    todo normalizado.
    """
    
    img_path = REY_SCAN_DIR / fila["forig"]
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        raise ValueError("No se pudo leer: " + str(img_path))
    
    alto_img, ancho_img = img.shape
    
    x = float(fila["x"])
    y = float(fila["y"])
    h = float(fila["h"])
    w = float(fila["w"])
    
    x_center = x + w / 2.0
    y_center = y + h / 2.0
    
    x_center_norm = x_center / ancho_img
    y_center_norm = y_center / alto_img
    w_norm = w / ancho_img
    h_norm = h / alto_img
    
    # Por seguridad, recortamos a [0,1]
    x_center_norm = min(max(x_center_norm, 0), 1)
    y_center_norm = min(max(y_center_norm, 0), 1)
    w_norm = min(max(w_norm, 0), 1)
    h_norm = min(max(h_norm, 0), 1)
    
    return x_center_norm, y_center_norm, w_norm, h_norm

In [ ]:
def preparar_split(df_split, nombre_split):
    """
    Copia imágenes y crea labels .txt para un split:
    train, val o test.
    """
    
    img_out_dir = YOLO_REY_DIR / "images" / nombre_split
    label_out_dir = YOLO_REY_DIR / "labels" / nombre_split
    
    for i, fila in df_split.iterrows():
        img_path = REY_SCAN_DIR / fila["forig"]
        
        # Nombre final
        nombre_img = Path(fila["forig"]).name
        destino_img = img_out_dir / nombre_img
        
        # Copiar imagen
        shutil.copy2(img_path, destino_img)
        
        # Crear label YOLO
        x_c, y_c, w, h = bbox_a_yolo(fila)
        
        # El label debe llamarse igual que la imagen, pero .txt
        label_name = Path(nombre_img).with_suffix(".txt").name
        label_path = label_out_dir / label_name
        
        # Clase 0 = figura de Rey
        linea = f"0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n"
        
        with open(label_path, "w", encoding="utf-8") as f:
            f.write(linea)

preparar_split(df_train, "train")
preparar_split(df_val, "val")
preparar_split(df_test, "test")

print("Dataset YOLO preparado.")

In [ ]:
def contar_archivos(carpeta, extensiones):
    total = 0
    
    for ext in extensiones:
        total += len(list(carpeta.glob(ext)))
    
    return total

for split in ["train", "val", "test"]:
    n_img = contar_archivos(YOLO_REY_DIR / "images" / split, ["*.jpg", "*.png"])
    n_lab = contar_archivos(YOLO_REY_DIR / "labels" / split, ["*.txt"])
    
    print(split, "imagenes:", n_img, "labels:", n_lab)

In [ ]:
yaml_path = YOLO_REY_DIR / "rey_bbox.yaml"

contenido_yaml = f"""
path: {YOLO_REY_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: rey
"""

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(contenido_yaml)

print("YAML creado en:")
print(yaml_path)

print("\nContenido:")
print(contenido_yaml)

In [ ]:
ejemplo_label = list((YOLO_REY_DIR / "labels" / "train").glob("*.txt"))[0]

print("Ejemplo label:", ejemplo_label)

with open(ejemplo_label, "r", encoding="utf-8") as f:
    print(f.read())